# Model training
The previous notebook gave us one-hot features for the peptides (feature encoding). Here we train three models: logistic regression, random forest and one small one-dimensionnal CNN. Metrics such as accuracy and AUC are printed after each model but full eval (confusion matrix, ROC curve, per-class report) will be available in notebook 04. Trained models are saved to `outputs/models/`.

In [60]:
import numpy as np
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, roc_auc_score
import joblib

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

### Hyperparameters: Shared settings
- **`RANDOM_STATE = 42`**: seed for reproducibility
- **`class_weight='balanced'`**: About 70% of peptides are negative. Without this, models lean toward always predicting "negative." Balanced weights up-weight the minority (immunogenic) class instead of throwing away data.


In [61]:
DATA_DIR   = Path("..") / "data" / "processed"
OUTPUT_DIR = Path("..") / "outputs" / "models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

## Load features

In [62]:
data = np.load(DATA_DIR / "features.npz")
X_train, X_test = data["X_train"], data["X_test"]
y_train, y_test = data["y_train"], data["y_test"]

print(f"X_train: {X_train.shape}  y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}   y_test:  {y_test.shape}")
print(f"Train positive rate: {y_train.mean():.3f}  |  Test positive rate: {y_test.mean():.3f}")

X_train: (38081, 231)  y_train: (38081,)
X_test:  (9521, 231)   y_test:  (9521,)
Train positive rate: 0.297  |  Test positive rate: 0.297


## Class imbalance

The dataset is imbalanced since it has a 2.49:1 ratio of negative peptides to positives (~70% label 0, ~30% label 1). This can lead to the model predicting negative most of the time and being right without necessarily finding the true positives. To fix this, all three models use `class_weight='balanced'`, which up-weights the minority class proportionally such that no samples are discarded. Class-weighting is appropriate here since resampling is typically reserved for ratios above 10:1.

In [63]:
classes = np.array([0, 1])
weights = compute_class_weight("balanced", classes=classes, y=y_train)
class_weight_dict = {0: weights[0], 1: weights[1]}
print(f"Class weights; 0: {weights[0]:.4f}  1: {weights[1]:.4f}")

Class weights; 0: 0.7114  1: 1.6825


## Model 1: Logistic Regression

Logistic regression is a linear baseline that produces calibrated probabilities and interpretable coefficients (one weight per amino acid position). It trains in seconds and sets a performance floor that the other models must beat.

- **`max_iter=1000`** : gives the solver enough iterations to converge on 231 features.
- **`solver='lbfgs'`** : standard choice for medium-sized dense problems like ours; stable and fast enough here.
- **`class_weight='balanced'`** : handles class imbalance (see above).
- *Defaults for everything else* : no regularization strength (`C=1.0`), which is fine as a first baseline.

**Why LR?** Simple, fast, interpretable. Reads coefficients as "which amino acid at which position pushes toward immunogenic." Good place to start with before trying a more complex model

In [64]:
lr = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    solver="lbfgs",
    random_state=RANDOM_STATE,
)
lr.fit(X_train, y_train)

lr_acc = accuracy_score(y_test, lr.predict(X_test))
lr_auc = roc_auc_score(y_test, lr.predict_proba(X_test)[:, 1])
print(f"LR Accuracy: {lr_acc:.4f}  AUC: {lr_auc:.4f}")

LR Accuracy: 0.5959  AUC: 0.6337


In [65]:
joblib.dump(lr, OUTPUT_DIR / "logistic_regression.joblib")
print("Saved logistic_regression.joblib")

Saved logistic_regression.joblib


## Model 2: Random Forest

Random forest is a non-linear ensemble that can capture position interactions or non linear pattern that the linear model misses. Train accuracy is printed alongside test accuracy to see if there's any overfitting. A large gap signals overfitting, which is common with deep unpruned trees on moderate-sized data.

- **`n_estimators=100`** : 100 decision trees averaged together
- **`class_weight='balanced'`** : same imbalance fix as LR.
- **`random_state=42`** : reproducible tree bootstraps.
- **`n_jobs=-1`**: uses all CPU cores
- *Left at defaults:* `max_depth=None`, `min_samples_leaf=1` : train accuracy hit **1.0** but test performance did not improve so much and was only modestly better, which is suggesting some overfitting.

**Why RF?** Can capture non-linear patterns that LR can’t. Tradeoff: harder to interpret, and ours overfit.

In [66]:
rf = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf.fit(X_train, y_train)

rf_train_acc = accuracy_score(y_train, rf.predict(X_train))
rf_acc       = accuracy_score(y_test,  rf.predict(X_test))
rf_auc       = roc_auc_score(y_test,   rf.predict_proba(X_test)[:, 1])
print(f"RF Train acc: {rf_train_acc:.4f}  Test acc: {rf_acc:.4f}  AUC: {rf_auc:.4f}")

RF Train acc: 1.0000  Test acc: 0.7523  AUC: 0.7024


AUC improved a bit here compared to LR, but it overfit heavily on training data.

In [67]:
joblib.dump(rf, OUTPUT_DIR / "random_forest.joblib")
print("Saved random_forest.joblib")

Saved random_forest.joblib


CNN coming in the next commit
